In [ ]:
import os
import logging
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from datasets import CubeObstacle, CylinderObstacle, BlockageDataset
from utils.config import Hyperparameters as hp
from utils.tools import calc_sig_strength_gpu

logging.basicConfig(level=logging.INFO)

def createDirectory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def save_df(path: str, name: str, data: list):
    if os.path.exists(f"{path}/{name}"):
        df = pd.read_csv(f"{path}/{name}")
        df = pd.concat([df, pd.DataFrame(data, columns=["gnd1_x", "gnd1_y", "gnd1_z", 
                                                        "gnd2_x", "gnd2_y", "gnd2_z",
                                                        "gnd3_x", "gnd3_y", "gnd3_z", ""
                                                        "gnd4_x", "gnd4_y", "gnd4_z",
                                                        "result_x", "result_y", "result_z"])])
    else:
        df = pd.DataFrame(data, columns=["gnd1_x", "gnd1_y", "gnd1_z", 
                                         "gnd2_x", "gnd2_y", "gnd2_z",
                                         "gnd3_x", "gnd3_y", "gnd3_z", 
                                         "gnd4_x", "gnd4_y", "gnd4_z",
                                         "result_x", "result_y", "result_z"])
    createDirectory(path)
    df.to_csv(f"{path}/{name}", index=False)

if __name__ == "__main__":
    result_ls = []
    
    obstacle_ls = [
        CubeObstacle(-30, 25, 35, 60, 20, 0.1),
        CubeObstacle(-30, -25, 45, 10, 35, 0.1),
        CubeObstacle(-30, -60, 35, 60, 20, 0.1),
        CubeObstacle(50, -20, 35, 25, 25, 0.1),
        CylinderObstacle(10, -5,  70, 15, 0.1),
    ]
    grid_step = 0.1
    
    test_height_ls = [50, 60, 80, 90, 100]
    for test_height in test_height_ls:
        logging.info(f"Test Height: {test_height}")
        dataset = BlockageDataset(20000, obstacle_ls, 4, dtype=torch.float32, grid_step=grid_step, height=test_height).to(hp.device)
        df = pd.read_csv('data/gn_coords_4.csv', header=None)
        dataset.gnd_nodes = torch.tensor(df.values, dtype=torch.float32, device=hp.device).reshape(-1, 4, 3)
        logging.info(f"len(dataset): {len(dataset)}")
        
        dataloader = DataLoader(dataset, batch_size=1)
        
        try:
            for i, data in enumerate(tqdm(dataloader)):
                station_pos, gnd_nodes, obst_points = data
                station_pos = station_pos.squeeze(0)
                gnd_nodes = gnd_nodes.squeeze(0)
                obst_points = obst_points.squeeze(0)

                chunk_size = 45000
                sig_chunks = []
                for j in range(0, station_pos.shape[0], chunk_size):
                    station_chunk = station_pos[j:j+chunk_size]  # [chunk_size, 3]
                    sig_chunk = calc_sig_strength_gpu(station_chunk, gnd_nodes, obst_points)
                    sig_chunks.append(sig_chunk)
                sig = torch.cat(sig_chunks, dim=0)
                sig = sig.reshape(dataset.grid_shape[0], dataset.grid_shape[1])

                max_idx = torch.unravel_index(torch.argmax(sig), sig.shape)
                np_gnd_nodes = gnd_nodes.cpu().numpy()
                np_max_pos = (max_idx[0].cpu().numpy()*grid_step, max_idx[1].cpu().numpy()*grid_step, test_height)
                logging.info(f"Max Signal: {sig[max_idx]}, Index: {max_idx}")

                result_ls.append([np_gnd_nodes[0, 0], np_gnd_nodes[0, 1], np_gnd_nodes[0, 2],
                                np_gnd_nodes[1, 0], np_gnd_nodes[1, 1], np_gnd_nodes[1, 2],
                                np_gnd_nodes[2, 0], np_gnd_nodes[2, 1], np_gnd_nodes[2, 2],
                                np_gnd_nodes[3, 0], np_gnd_nodes[3, 1], np_gnd_nodes[3, 2],
                                np_max_pos[0]-(hp.area_size//2), np_max_pos[1]-(hp.area_size//2), np_max_pos[2]])
                logging.info(f"Result: {max_idx}, sig_max: {sig[max_idx]}")

        except KeyboardInterrupt as e:
            logging.warning("Interrupted by user: " + str(e))
        finally:
            save_df("data/result_plots/height", f"height{test_height}_data.csv", result_ls)


INFO:root:Test Height: 50
100%|██████████| 20000/20000 [00:00<00:00, 89834.85it/s]
INFO:root:len(dataset): 20000
  0%|          | 0/20000 [00:00<?, ?it/s]INFO:root:Max Signal: 14.001489639282227, Index: (tensor(426, device='cuda:0'), tensor(1360, device='cuda:0'))
INFO:root:Result: (tensor(426, device='cuda:0'), tensor(1360, device='cuda:0')), sig_max: 14.001489639282227
  0%|          | 1/20000 [00:07<40:56:30,  7.37s/it]INFO:root:Max Signal: 13.353199005126953, Index: (tensor(139, device='cuda:0'), tensor(400, device='cuda:0'))
INFO:root:Result: (tensor(139, device='cuda:0'), tensor(400, device='cuda:0')), sig_max: 13.353199005126953
  0%|          | 2/20000 [00:14<39:20:38,  7.08s/it]INFO:root:Max Signal: 13.264568328857422, Index: (tensor(1752, device='cuda:0'), tensor(419, device='cuda:0'))
INFO:root:Result: (tensor(1752, device='cuda:0'), tensor(419, device='cuda:0')), sig_max: 13.264568328857422
  0%|          | 3/20000 [00:21<38:49:54,  6.99s/it]INFO:root:Max Signal: 13.6452569